In [1]:
import json

with open("../data/sequences.json", "r") as f:
    all_sequences = json.load(f)

print(len(all_sequences))

2850


In [2]:
import pandas as pd

df = pd.read_csv("../data/spotify_tracks.csv")

In [3]:
mood_features = [
    "energy",
    "valence",
    "danceability",
    "acousticness",
    "tempo"
]

In [4]:
import numpy as np

X = []
y = []

for seq in all_sequences:

    feature_sequence = (
        df.loc[seq, mood_features]
        .values
    )

    X.append(feature_sequence[:-1])
    y.append(feature_sequence[-1])

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(2850, 7, 5)
(2850, 5)


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
model = Sequential()

model.add(
    GRU(
        128,
        return_sequences=True,
        input_shape=(
            X_train.shape[1],
            X_train.shape[2]
        )
    )
)

model.add(Dropout(0.2))

model.add(
    GRU(
        64
    )
)

model.add(Dense(32, activation='relu'))

model.add(Dense(y_train.shape[1]))

h:\project\mood-sequencer-rnn\tf_env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss='mse',
    metrics=['mae']
)

In [22]:
history = model.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=16,
    validation_data=(
        X_test,
        y_test
    ),
    callbacks=[early_stop]
)

Epoch 1/200
143/143 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 13.4344 - mae: 1.4289 - val_loss: 11.3600 - val_mae: 1.2528
Epoch 2/200
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 14.4379 - mae: 1.4681 - val_loss: 11.7322 - val_mae: 1.2887
Epoch 3/200
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 13.3605 - mae: 1.4170 - val_loss: 11.8033 - val_mae: 1.2971
Epoch 4/200
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 13.4790 - mae: 1.4224 - val_loss: 11.6935 - val_mae: 1.2794
Epoch 5/200
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 13.4474 - mae: 1.4164 - val_loss: 11.2360 - val_mae: 1.2635
Epoch 6/200
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 13.8294 - mae: 1.4319 - val_loss: 12.5500 - val_mae: 1.3747
Epoch 7/200
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 13.0869 - mae: 1.4042 - val_loss: 11.9700 - val_mae: 1.2951
Epoch 8/200
143/143 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 13.0910 - mae: 1.4032 - val_loss: 11.5330 - val_mae: 1.3029
Epoch 9/200
143/143 ━━━━

In [23]:
loss, mae = model.evaluate(
    X_test,
    y_test
)

print("Test Loss:", loss)
print("Test MAE:", mae)

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 11.3600 - mae: 1.2528
Test Loss: 11.359997749328613
Test MAE: 1.252807855606079
